# 06.5 Dynamic topic modeling con BERTopic (Híbrido)

Este cuaderno prepara un flujo para modelar temas dinámicos sobre los posteos de Revista Anfibia usando una arquitectura híbrida:
- **Embeddings/Clustering:** Usa el texto original para capturar la sintaxis y semántica.
- **Representación (Palabras clave):** Usa el texto lematizado para evitar redundancia morfológica en español.

In [2]:
# EJECUTAR ESTA CELDA PRIMERO QUE NADA (Y REINICIAR EL KERNEL ANTES SI FALLÓ)
import coverage.types as ct

# Parche de compatibilidad para evitar que Numba/UMAP crasheen
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

## 1. Configuración del entorno

In [3]:
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Rutas
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
assert DATA_PATH.exists(), f"Falta archivo de entrada: {DATA_PATH.name}"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_hibrido"
GRAFICOS_DIR = OUTPUT_DIR / "graficos"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
GRAFICOS_DIR.mkdir(exist_ok=True, parents=True)

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga y normalización de metadatos

In [4]:
MAX_DOCUMENTS = 4000
USE_COLUMNS = ["ID", "source_url", "title_text", "tagline", "autor_names", "post_date", "category", "text_type", "document_text_original", "document_text_final"]

df = pd.read_csv(DATA_PATH, usecols=USE_COLUMNS, nrows=MAX_DOCUMENTS)

# Llenar vacíos
df["document_text_original"] = df["document_text_original"].fillna("").astype(str).str.strip()
df["document_text_final"] = df["document_text_final"].fillna("").astype(str).str.strip()

# Filtrar vacíos
df = df[df["document_text_original"].ne("") & df["document_text_final"].ne("")].copy()
df = df.reset_index(drop=True)
df.insert(0, "doc_id", df.index)

print(f"Documentos listos: {df.shape[0]}")

# Normalizar categorías
def split_categories(value):
    parts = re.split(r"[;,]", str(value))
    return [part.strip() for part in parts if part.strip()]

category_rows = []
for doc_id, value in zip(df["doc_id"], df["category"]):
    categories = split_categories(value)
    if not categories:
        categories = ["Sin categoria"]
    for category in categories:
        category_rows.append({"doc_id": doc_id, "category_normalized": category})

categories_df = pd.DataFrame(category_rows)
category_counts = categories_df["category_normalized"].value_counts().rename_axis("category").reset_index(name="documents")

Documentos listos: 3556


## 3. Preparación de Stopwords y Textos

Definimos las palabras vacías generales del español y las específicas del dominio periodístico de Revista Anfibia. 
Separamos explícitamente el corpus en dos listas: una con el texto original (para la semántica) y otra con el texto lematizado (para la frecuencia de palabras).

In [56]:
SPANISH_STOPWORDS = {
    "a", "acá", "ahi", "ahí", "al", "algo", "algunas", "algunos", "ante", "antes", "aquel", "aquella", "aquellas", "aquello", "aquellos",
    "aqui", "aquí", "arriba", "asi", "así", "atras", "aun", "aunque", "bajo", "bien", "cada", "casi", "como", "con", "contra",
    "cual", "cuando", "de", "del", "desde", "donde", "dos", "durante", "e", "el", "él", "ella", "ellas", "ellos", "en", "entre",
    "era", "eran", "eres", "es", "esa", "esas", "ese", "eso", "esos", "esta", "está", "estaba", "estaban", "estado", "estamos", "estan", "están",
    "estar", "este", "esto", "estos", "estoy", "fue", "fueron", "fui", "ha", "haber", "habia", "había", "hace", "hacen", "hacer", "hacia",
    "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "más", "me", "mi", "mis", "mismo", "mucha", "muchas", "mucho", "muchos",
    "muy", "nada", "ni", "no", "nos", "nosotras", "nosotros", "nuestra", "nuestras", "nuestro", "nuestros", "o", "otra", "otras", "otro", "otros",
    "para", "pero", "poco", "por", "porque", "que", "qué", "quien", "quién", "se", "sea", "segun", "según", "ser", "si", "sí", "sin",
    "sobre", "son", "su", "sus", "tambien", "también", "tan", "tanto", "te", "tenia", "tenía", "tener", "tiene", "tienen", "todo", "todos",
    "tu", "tus", "un", "una", "unas", "uno", "unos", "usted", "ustedes", "va", "van", "vez", "y", "ya", "yo",
}

DOMAIN_STOPWORDS = {
    "anfibia", "revista", "cronica", "crónica", "texto", "nota", "parte", "dice", "dijo", "hacer", "hace", "anos", "años",
    "argentina", "argentino", "argentinas", "argentinos", "buenos", "aires", "ano", "año", "poder",
    "decir", "ir", "ver", "solo", "llegar", "saber", "dar", "querer", "pasar", "volver", "día", "nuevo", "parecer", "quedar", 
    "creer", "dejar", "seguir", "encontrar", "llamar", "venir", "empezar", "persona", "historia", "vida", "caso", "momento", 
    "tiempo", "lugar", "forma", "primero", "despues", "después", "hoy", "ahora", "siempre", "nunca", "mientras", "cosa", 
    "alguno", "algun", "medio", "propio", "cierto", "grande"
}

NUEVAS_STOPWORDS = {
    # Verbos comodín y acciones genéricas
    "hablar", "deber", "pensar", "poner", "llevar", "vivir", "contar", "salir", 
    "tomar", "buscar", "trabajar", "tratar", "terminar", "sentir", "conocer", 
    "mostrar", "pedir", "preguntar", "mirar", "escuchar", "entender", "permitir", 
    "lograr", "ganar", "existir", "aparecer", "convertir", "explicar", "esperar", 
    "comenzar", "recordar", "generar", "cambiar", "construir", "recibir", "producir", 
    "usar", "abrir", "sacar", "presentar", "necesitar", "escribir", "jugar",
    
    # Adverbios, pronombres, números y conectores
    "cómo", "sino", "tú", "último", "menos", "tres", "sólo", "frente", "entonces", 
    "luego", "manera", "gran", "lado", "mil", "mes", "mejor", "además", "único", 
    "distinto", "varios", "tipo", "sentido", "junto", "ninguno", "tal", "modo", 
    "aún", "posible", "cualquiera", "nadie", "incluso", "embargo", "buen", 
    "importante", "largo", "allí", "alto", "tarde", "general", "partir", "pasado", 
    "claro", "todavía", "allá", "final", "medida", "dentro", "paso", "cuatro", "segundo",
    
    # Sustantivos genéricos y meta-periodismo
    #"público","trabajo", "libro","discurso", "obra"
    "grupo", "hora", "espacio", "gente", "problema", 
    "calle", "relación", "hecho", "palabra", "punto", "foto", "proceso", "tema", 
    "ejemplo", "imagen", "cuenta",  "situación", "través", 
    "mayoría", "pregunta", "diario", "acción", "posibilidad", "nombre", "fin", "centro"
}

NUEVAS_STOPWORDS_2 = {
    # Verbos en infinitivo
    #"leer", "morir", "pagar"
    "perder", "entrar", "decidir", "intentar", "sostener", 
    "mantener", "responder", "gustar", "considerar", "caer", 
    "suceder", "resultar", "ocurrir", "sumar", "reconocer", "alcanzar", 
    "soler", "proponer", "incluir", "tocar", "crear",
    
    # Adjetivos, adverbios, pronombres y números
    "mayor", "principal", "viejo", "pequeño", "común", "diferente", "igual", 
    "mal", "necesario", "fuerte", "difícil", "real", "apenas", "tampoco", 
    "cinco", "diez", "veces", "alguien", "mañana", "verdad",
    
    # Sustantivos abstractos, de relleno o métricas
    "idea", "mano", "amigo", "experiencia", "millón", "condición", "respuesta", 
    "diferencia", "camino", "puerta", "acuerdo", "resultado", "lucha", 
    "cuestión", "práctica", "presente", "ojo", "efecto", "cara", "personal", 
    "valor", "término"
}

NUEVAS_STOPWORDS_3 = {
    # Verbos en infinitivo restantes
    "funcionar", "ofrecer", "compartir", "comprar", "cumplir", "elegir", 
    "realizar", "crecer", "conseguir", "formar", "ocupar", "nacer", "correr",
    
    # Gramática, números y tiempo
    "cuyo", "cerca", "actual", "20", "ciento", "minuto", 
    
    # Comodines periodísticos o palabras vacías
    "clave", "llamado", "base", "falta"
}

NUEVAS_STOPWORDS_4 = {
    # Pronombres, preposiciones y conectores
    "ambos", "pesar", "fuera", "resto", "vuelta", "razón", 
    
    # Verbos en infinitivo genéricos
    #"publicar", "aprender", "participar",
    
    # Sustantivos abstractos de uso métrico/estructural
    #"número", 
    "uso",

    # Preposición temporal/espacial
    "tras", 
    
    # Verbos comodín / muletillas de redacción
    "acompañar", "señalar"
}



# Actualizás tu lista final así:
DOMAIN_STOPWORDS = DOMAIN_STOPWORDS.union(NUEVAS_STOPWORDS)
DOMAIN_STOPWORDS = DOMAIN_STOPWORDS.union(NUEVAS_STOPWORDS_2)
DOMAIN_STOPWORDS = DOMAIN_STOPWORDS.union(NUEVAS_STOPWORDS_3)
DOMAIN_STOPWORDS = DOMAIN_STOPWORDS.union(NUEVAS_STOPWORDS_4)

STOPWORDS = sorted(SPANISH_STOPWORDS | DOMAIN_STOPWORDS)

# Preparamos las dos versiones
docs_naturales = df["document_text_original"].tolist()
docs_lematizados = df["document_text_final"].tolist()

# Fechas
timestamps = pd.to_datetime(df["post_date"].str.split(' ').str[0], format="%Y-%m-%d", errors="coerce")

## 4. Entrenamiento del Modelo Híbrido

Acá ocurre la separación de la arquitectura:
1. `SentenceTransformer` calcula los embeddings usando el **texto natural**.
2. `CountVectorizer` construye el vocabulario usando los **lemas**.
3. `BERTopic.fit_transform()` recibe los textos lematizados para el c-TF-IDF, pero se le inyectan los `embeddings_naturales` precalculados para que el clustering (UMAP + HDBSCAN) se haga sobre el espacio semántico original.

In [18]:
# 1. Embeddings sobre texto crudo
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings_naturales = embedding_model.encode(docs_naturales, show_progress_bar=True)

# 2. Vectorizador sobre texto lematizado
vectorizer_model = CountVectorizer(stop_words=STOPWORDS, min_df=3)
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="spanish",
    calculate_probabilities=True,
    min_topic_size=10,
    nr_topics="auto",
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
)

# 3. Entrenamiento: Clusteriza con embeddings naturales, extrae palabras con lemas
try:
    topic_model.load(OUTPUT_DIR / "bertopic_hibrido_model")
    topics, probs = topic_model.transform(docs_lematizados, embeddings=embeddings_naturales)
    print("Modelo cargado desde disco.")
except:
    topics, probs = topic_model.fit_transform(docs_lematizados, embeddings=embeddings_naturales)
    topic_model.save(OUTPUT_DIR / "bertopic_hibrido_model")
    print("Modelo híbrido entrenado y guardado.")

topic_info = topic_model.get_topic_info()
display(topic_info.head(10))

Batches: 100%|██████████| 112/112 [01:35<00:00,  1.18it/s]
2026-08-27 21:43:15,181 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Modelo híbrido entrenado y guardado.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1624,-1_político_política_periodista_hablar,"[político, política, periodista, hablar, socie...",[el antropóloga que incomodar como uno estrell...
1,0,1800,0_existir_pensar_sociedad_tratar,"[existir, pensar, sociedad, tratar, además, gé...",[del escrache a el pedagogía del deseo a dos a...
2,1,46,1_papal_catolicismo_arzobispo_católica,"[papal, catolicismo, arzobispo, católica, cató...",[el último hombre bueno hacer 12 año cuando jo...
3,2,37,2_soviética_soviético_vladimir_rusia,"[soviética, soviético, vladimir, rusia, ucrani...",[el paz ser uno sueño eterno a uno año del con...
4,3,19,3_israel_netanyahu_israelí_palestina,"[israel, netanyahu, israelí, palestina, palest...",[herida de uno cuento inventado cada vez más v...
5,4,18,4_astrónomo_ciencia_astrología_científico,"[astrónomo, ciencia, astrología, científico, u...",[el detector de partícula el físico alberto et...
6,5,12,5_paciente_médico_morir_clínico,"[paciente, médico, morir, clínico, muerte, mor...",[el curva no ser uno infografía ser uno cicatr...


In [7]:
topics_over_time = topic_model.topics_over_time(
    docs_lematizados,
    timestamps.tolist(),
    datetime_format="%Y-%m-%d",
    nr_bins=20,
)
topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time.csv", index=False)

analysis_df = df[['post_date', 'category']].copy()
analysis_df['category'] = categories_df['category_normalized']
analysis_df['category'] = analysis_df['category'].fillna('sin_categoria').astype(str).str.strip().replace('', 'sin_categoria')
analysis_df['post_date'] = pd.to_datetime(analysis_df['post_date'], errors='coerce')
analysis_df['topic'] = topics

topics_by_category = (
    analysis_df
    .assign(date=lambda x: x['post_date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['date', 'category', 'topic'])
    .size()
    .reset_index(name='count')
)
topics_by_category = topics_by_category[topics_by_category['topic'] != -1].copy()
topics_by_category.to_csv(OUTPUT_DIR / 'topics_over_time_by_category.csv', index=False)

## 5. Extracción de la evolución temporal

In [8]:
fig = topic_model.visualize_barchart(top_n_topics=10, n_words=8, title="<b>Palabras representativas (Híbrido)</b>")
colors = px.colors.qualitative.G10
for i, trace in enumerate(fig.data):
    trace.marker.color = colors[i % len(colors)]

fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
fig.update_layout(height=750, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

fig.write_html(GRAFICOS_DIR / "01_barchart_topicos.html")
fig.show()

In [11]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Matriz de coocurrencia de términos en el corpus
vectorizer = CountVectorizer(vocabulary=topic_model.vectorizer_model.vocabulary_)
X = vectorizer.transform(docs_lematizados)
binary_matrix = (X > 0).astype(int)

doc_counts = np.array(binary_matrix.sum(axis=0)).flatten()
n_docs = binary_matrix.shape[0]

# 2. Función para calcular coherencia NPMI por tópico
def calculate_topic_npmi(words, top_n=10):
    words = words[:top_n]
    vocab = vectorizer.vocabulary_
    indices = [vocab[w] for w in words if w in vocab]
    
    if len(indices) < 2:
        return 0.0
    
    npmi_scores = []
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            idx_i, idx_j = indices[i], indices[j]
            p_i = doc_counts[idx_i] / n_docs
            p_j = doc_counts[idx_j] / n_docs
            
            # Documentos donde coocurren ambas palabras
            col_i = binary_matrix[:, idx_i].toarray().flatten()
            col_j = binary_matrix[:, idx_j].toarray().flatten()
            cooc = np.sum(col_i & col_j)
            p_ij = cooc / n_docs
            
            if p_ij > 0:
                pmi = np.log2(p_ij / (p_i * p_j))
                npmi = pmi / (-np.log2(p_ij))
                npmi_scores.append(npmi)
            else:
                npmi_scores.append(-1.0)
                
    return np.mean(npmi_scores) if npmi_scores else 0.0

# 3. Métricas consolidadas por tópico
topic_metrics = []
topics_dict = topic_model.get_topics()

for t_id, words_weights in topics_dict.items():
    if t_id == -1:
        continue
    words = [w for w, _ in words_weights]
    count = topic_model.get_topic_info(t_id)["Count"].values[0]
    pct = (count / n_docs) * 100
    npmi = calculate_topic_npmi(words)
    
    topic_metrics.append({
        "Topic": t_id,
        "Name": ", ".join(words[:4]),
        "Docs": count,
        "% Corpus": round(pct, 2),
        "Coherencia_NPMI": round(npmi, 4)
    })

metrics_df = pd.DataFrame(topic_metrics).sort_values(by="Docs", ascending=False)
display(metrics_df.head(15))

,Topic,Name,Docs,% Corpus,Coherencia_NPMI
0,0,"política, político, gobierno, discurso",1218,34.25,0.1509
1,1,"brasil, futbolista, fifa, fútbol",143,4.02,0.3029
2,2,"pedagógico, enseñanza, educativo, escolar",111,3.12,0.2686
3,3,"aborto, abortar, anticonceptivo, embarazo",81,2.28,0.2436
4,4,"vacunar, vacuna, epidemia, vacunación",70,1.97,0.3803
5,5,"mediático, periodismo, noticia, digital",53,1.49,0.0982
6,6,"papal, catolicismo, arzobispo, católica",47,1.32,0.4778
7,7,"arte, art, estética, artista",33,0.93,0.2134
8,8,"fotoperiodismo, fotógrafo, fotográfico, fotogr...",28,0.79,0.1634
9,9,"sindicalista, sindicalismo, sindical, político",23,0.65,0.1895


* **% Corpus (% del total de notas):**
  * **$< 0.5\% - 0.8\%$:** Tópicos marginales o notas aisladas.
  * **$\ge 1.0\%$:** Tópicos con volumen representativo dentro del corpus.

* **Coherencia NPMI (rango de -1 a +1):**
Mide qué tan fuertemente asociadas están dos palabras en un corpus.
  * **$> 0.10$:** Excelente coherencia semántica. Las palabras clave aparecen frecuentemente juntas en los mismos contextos.
  * **$0.00$ a $0.10$:** Aceptable. Tópico amplio o general, pero interpretable.
  * **$< -0.05$:** Tópico ruidoso o inconexo. Candidato directo a descartar o fusionar.

## 6. Diagnóstico: El problema de los Outliers (-1)
Por defecto, el algoritmo de clustering de BERTopic (HDBSCAN) es muy estricto con la densidad. Si un artículo es un poco atípico o híbrido, en lugar de forzarlo a un grupo, lo expulsa y lo etiqueta como ruido (Tópico `-1`). Veamos qué impacto tiene esto en nuestro corpus.

In [19]:
import plotly.graph_objects as go

outliers_base = topics.count(-1)
clasificados_base = len(topics) - outliers_base
pct_ruido = (outliers_base / len(topics)) * 100

fig_diag = go.Figure(data=[go.Pie(
    labels=["Ruido (-1) / Sin clasificar", "Tópicos Válidos"], 
    values=[outliers_base, clasificados_base],
    hole=.4,
    marker=dict(colors=["#d62728", "#2ca02c"])
)])

fig_diag.update_layout(
    title=f"<b>Cobertura (HDBSCAN puro)</b><br><sup>El modelo descartó el {pct_ruido:.1f}% de las notas.</sup>",
    width=600, height=450,
    template="plotly_white"
)
fig_diag.show()

### 6.1 Solución: Redistribución por similitud de Embeddings
Tener la mitad del archivo sin clasificar no es viable para un sistema editorial. Para solucionarlo, forzamos al modelo a tomar todas las notas `-1` y buscar su centroide más cercano utilizando la distancia del coseno sobre los embeddings originales.

In [63]:
# ==============================================================================
# 1. ENTRENAMIENTO Y REDUCCIÓN DE OUTLIERS (Instancia Única)
# ==============================================================================
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

vectorizer_model = CountVectorizer(stop_words=STOPWORDS, min_df=3)
representation_model = KeyBERTInspired()
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

topic_model = BERTopic(
    umap_model=umap_model,
    embedding_model=embedding_model,
    language="spanish",
    calculate_probabilities=True,
    min_topic_size=10,
    nr_topics=None,
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
)

print("1. Entrenando modelo híbrido base desde cero...")
topics, probs = topic_model.fit_transform(docs_lematizados, embeddings=embeddings_naturales)

outliers_base = topics.count(-1)
clasificados_base = len(topics) - outliers_base
pct_ruido = (outliers_base / len(topics)) * 100

print(f"\n2. Reasignando notas huérfanas (-1)... Ruido inicial: {pct_ruido:.1f}%")
nuevos_topics = topic_model.reduce_outliers(
    docs_lematizados,
    topics,
    strategy="embeddings",
    embeddings=embeddings_naturales
)

print("\n3. Recalibrando palabras clave (c-TF-IDF)...")
topic_model.update_topics(
    docs_lematizados,
    topics=nuevos_topics,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

topic_model.save(OUTPUT_DIR / "bertopic_hibrido_model") 
topics = nuevos_topics
print("\n✅ Modelo optimizado y guardado exitosamente.")

# ==============================================================================
# 2. DIAGNÓSTICO VISUAL DE OUTLIERS
# ==============================================================================
outliers_nuevos = topics.count(-1)
clasificados_nuevos = len(topics) - outliers_nuevos

fig_diag = go.Figure(data=[go.Pie(
    labels=["Ruido (-1) / Sin clasificar", "Tópicos Válidos"], 
    values=[outliers_nuevos, clasificados_nuevos],
    hole=.4,
    marker=dict(colors=["#d62728", "#2ca02c"])
)])

fig_diag.update_layout(
    title=f"<b>Cobertura optimizada</b><br><sup>Outliers reducidos a {outliers_nuevos} notas.</sup>",
    width=600, height=450,
    template="plotly_white"
)
fig_diag.show()

# ==============================================================================
# 3. EVOLUCIÓN TEMPORAL (Topics Over Time)
# ==============================================================================
print("\nCalculando evolución temporal de los tópicos...")
topics_over_time = topic_model.topics_over_time(
    docs=docs_lematizados,
    timestamps=timestamps.tolist(),
    datetime_format="%Y-%m-%d",
    nr_bins=20,
)

topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time.csv", index=False)

analysis_df = df[['post_date', 'category']].copy()
analysis_df['category'] = categories_df['category_normalized']
analysis_df['category'] = analysis_df['category'].fillna('sin_categoria').astype(str).str.strip().replace('', 'sin_categoria')
analysis_df['post_date'] = pd.to_datetime(analysis_df['post_date'], errors='coerce')
analysis_df['topic'] = topics

topics_by_category = (
    analysis_df
    .assign(date=lambda x: x['post_date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['date', 'category', 'topic'])
    .size()
    .reset_index(name='count')
)

topics_by_category.to_csv(OUTPUT_DIR / 'topics_over_time_by_category.csv', index=False)
print("-> Archivos temporales guardados en outputs.")


1. Entrenando modelo híbrido base desde cero...


2026-08-27 22:28:01,814 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



2. Reasignando notas huérfanas (-1)... Ruido inicial: 40.6%

3. Recalibrando palabras clave (c-TF-IDF)...


2026-08-27 22:28:21,102 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.



✅ Modelo optimizado y guardado exitosamente.



Calculando evolución temporal de los tópicos...
-> Archivos temporales guardados en outputs.


In [87]:
fig = topic_model.visualize_barchart(top_n_topics=16, n_words=8, title="<b>Palabras representativas (Híbrido)</b>")
colors = px.colors.qualitative.G10
for i, trace in enumerate(fig.data):
    trace.marker.color = colors[i % len(colors)]

fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
fig.update_layout(height=850, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

fig.write_html(GRAFICOS_DIR / "01_barchart_topicos.html")
fig.show()

Analizar los nuevos topicos

La coherencia NPMI (Normalized Pointwise Mutual Information o Información Mutua Puntual Normalizada) es una métrica estadística que se usa en procesamiento de lenguaje natural para evaluar qué tan coherentes y lógicas son las palabras clave que definen un tópico en modelos como BERTopic o LDA.

¿Cómo funciona por dentro?
Información Mutua (PMI): Mide la probabilidad de que la Palabra A y la Palabra B aparezcan juntas en un mismo documento, en comparación con la probabilidad de que aparezcan separadas por casualidad. Si aparecen juntas mucho más de lo esperado, el puntaje es alto.

Normalización (N): El problema del PMI tradicional es que su valor cambia según la frecuencia de las palabras (las palabras rarísimas daban números extraños). Al "normalizarlo", se divide el resultado por un factor de máxima entropía, logrando que el indicador siempre se mueva estrictamente en una escala fija de -1 a +1.

In [76]:
# 1. Matriz de coocurrencia de términos en el corpus
vectorizer = CountVectorizer(vocabulary=topic_model.vectorizer_model.vocabulary_)
X = vectorizer.transform(docs_lematizados)
binary_matrix = (X > 0).astype(int)

doc_counts = np.array(binary_matrix.sum(axis=0)).flatten()
n_docs = binary_matrix.shape[0]

# 2. Función para calcular coherencia NPMI por tópico
def calculate_topic_npmi(words, top_n=10):
    words = words[:top_n]
    vocab = vectorizer.vocabulary_
    indices = [vocab[w] for w in words if w in vocab]
    
    if len(indices) < 2:
        return 0.0
    
    npmi_scores = []
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            idx_i, idx_j = indices[i], indices[j]
            p_i = doc_counts[idx_i] / n_docs
            p_j = doc_counts[idx_j] / n_docs
            
            # Documentos donde coocurren ambas palabras
            col_i = binary_matrix[:, idx_i].toarray().flatten()
            col_j = binary_matrix[:, idx_j].toarray().flatten()
            cooc = np.sum(col_i & col_j)
            p_ij = cooc / n_docs
            
            if p_ij > 0:
                pmi = np.log2(p_ij / (p_i * p_j))
                npmi = pmi / (-np.log2(p_ij))
                npmi_scores.append(npmi)
            else:
                npmi_scores.append(-1.0)
                
    return np.mean(npmi_scores) if npmi_scores else 0.0

# 3. Métricas consolidadas por tópico
topic_metrics = []
topics_dict = topic_model.get_topics()

for t_id, words_weights in topics_dict.items():
    if t_id == -1:
        continue
    words = [w for w, _ in words_weights]
    count = topic_model.get_topic_info(t_id)["Count"].values[0]
    pct = (count / n_docs) * 100
    npmi = calculate_topic_npmi(words)
    
    topic_metrics.append({
        "Topic": t_id,
        "Name": ", ".join(words[:4]),
        "Docs": count,
        "% Corpus": round(pct, 2),
        "Coherencia_NPMI": round(npmi, 4)
    })

metrics_df = pd.DataFrame(topic_metrics).sort_values(by="Docs", ascending=False)
display(metrics_df.head(15))

,Topic,Name,Docs,% Corpus,Coherencia_NPMI
0,0,"político, política, electoral, candidato",472,13.27,0.2639
3,3,"literario, escritora, autor, literatura",240,6.75,0.3031
1,1,"feminismo, feminista, patriarcado, masculinidad",217,6.10,0.2869
6,6,"asesinato, policía, policial, maldonado",200,5.62,0.2183
2,2,"fifa, futbolista, fútbol, deporte",153,4.30,0.3681
4,4,"déficit, deuda, endeudamiento, inflación",152,4.27,0.3114
5,5,"pedagógico, enseñanza, escolar, educativo",132,3.71,0.2686
11,11,"asesino, testimonio, testigo, víctima",96,2.70,0.2465
14,14,"mediático, periodístico, político, periodista",95,2.67,0.1176
7,7,"aborto, abortar, anticonceptivo, embarazo",91,2.56,0.2436


In [79]:
fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=15)
colors = px.colors.qualitative.G10

for i, t in enumerate(list(fig.data)):
    c = colors[i % len(colors)]
    vis = True if i < 3 else 'legendonly'
    name = t.name.replace("_", " ")

    t.update(name=name, visible=vis, legendgroup=f"g{i}", line=dict(color=c, width=3),
             hovertemplate=f"<b>{name}</b><br>Fecha: %{{x}}<br>Frecuencia: %{{y}}<extra></extra>")
    
    idx = list(t.y).index(max(t.y))
    fig.add_trace(go.Scatter(
        x=[t.x[idx]], y=[t.y[idx]], text=[t.y[idx]], mode="markers+text", textposition="top center",
        showlegend=False, visible=vis, legendgroup=f"g{i}",
        marker=dict(size=10, color=c, symbol="diamond", line=dict(width=1.5, color="#111")),
        hovertemplate=f"<b>Pico ({name})</b><br>Fecha: %{{x}}<br>Máximo: %{{y}}<extra></extra>"
    ))

fig.update_layout(title="<b>Evolución temporal de Tópicos</b>", plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=600, hovermode="closest")
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")

fig.write_html(GRAFICOS_DIR / "02_evolucion_topicos_picos.html")
fig.show()

In [84]:
# 1. Reducimos los embeddings a 2D explícitamente para graficar
reducer = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
coords_2d = reducer.fit_transform(embeddings_naturales)

# 2. Armamos un DataFrame limpio con toda la info de cada documento
topic_info = topic_model.get_topic_info()
topic_names = {row['Topic']: row['Name'] for _, row in topic_info.iterrows()}

plot_df = pd.DataFrame({
    'x': coords_2d[:, 0],
    'y': coords_2d[:, 1],
    'topic': topics,
    'title': df['title_text'].values
})

plot_df['topic_name'] = plot_df['topic'].map(topic_names)
plot_df['topic_label'] = plot_df['topic'].astype(str) + ": " + plot_df['topic_name']

# 3. ORDENAR LA LEYENDA
# Opción A: Ordenados estrictamente por número de Tópico (0, 1, 2, 3...)
# Si prefieres ordenarlos por frecuencia (del que más notas tiene al que menos), 
# cambiamos 'topic' por la frecuencia del topic_info.
topics_ordenados = sorted(plot_df['topic'].unique())
categorias_ordenadas = [
    f"{t}: {topic_names[t]}" for t in topics_ordenados
]

# Convertimos a categoría ordenada de Pandas para que la leyenda respete este orden
plot_df['topic_label'] = pd.Categorical(
    plot_df['topic_label'], 
    categories=categorias_ordenadas, 
    ordered=True
)

# 4. Creamos el gráfico con Plotly Express
fig = px.scatter(
    plot_df,
    x='x',
    y='y',
    color='topic_label',
    custom_data=['topic_label', 'title']
)

# 5. Personalizamos el hover
fig.update_traces(
    hovertemplate=(
        "<b>Tópico:</b> %{customdata[0]}<br>"
        "<b>Título:</b> %{customdata[1]}"
        "<extra></extra>"
    ),
    marker=dict(size=6, opacity=0.8)
)

# 6. Limpieza final de diseño
fig.update_layout(
    title="<b>Espacio de documentos (Embeddings Híbridos)</b>",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='#fff',
    height=700,
    hovermode="closest",
    xaxis=dict(showticklabels=False, zeroline=False, showgrid=False, title=""),
    yaxis=dict(showticklabels=False, zeroline=False, showgrid=False, title=""),
    legend=dict(title="<b>Tópicos</b>", orientation="v", y=1, x=1.02)
)

fig.write_html(GRAFICOS_DIR / "03_espacio_documentos.html")
fig.show()